# Visual evaluation — viewer

All the heavy work (training + figures) is done by `run_eval.py` on the cluster
(`sbatch run_eval.sh`). This notebook is only a **viewer**: it displays the PNGs
already written into `results/`, and optionally re-plots from the saved artifacts
if you want to tweak colors / FIDs (cheap — loads `pred.npy` / `model.joblib`,
not the embeddings).

Figures per model (`results/rf/`, `results/log_reg/`): `confusion.png`, `prf.png`,
`fidmaps.png`, `learning_curve.png`. Shared: `results/token_distribution.png`.

## A. Just show the generated PNGs

In [ ]:
from IPython.display import Image, Markdown, display

RESULT_DIRS = {"RandomForest": "results/rf", "LogisticRegression": "results/log_reg"}
FIGS = ["confusion.png", "prf.png", "fidmaps.png", "learning_curve.png"]

display(Markdown("### Token distribution (shared)"))
display(Image("results/token_distribution.png"))

for name, d in RESULT_DIRS.items():
    display(Markdown(f"### {name}"))
    for f in FIGS:
        display(Image(f"{d}/{f}"))

## B. (Optional) Re-plot interactively from the artifacts

Use this only if you want to change something (e.g. which FIDs to map). It does
**not** retrain — it loads the saved predictions / model. Building `ds` is cheap
(reads the shapefile and lists tile paths; does not load embeddings).

In [ ]:
import os
import numpy as np
import joblib
from seg_dataset import DisturbanceSegDataset
import eval_plots as ep

# confusion + per-class bars, rendered inline from saved predictions
for name, d in RESULT_DIRS.items():
    y_test = np.load(f"{d}/y_test.npy")
    pred = np.load(f"{d}/pred.npy")
    print(f"=== {name} ===  ({len(y_test)} test tokens)")
    ep.plot_confusion(y_test, pred, name, show=True)
    ep.plot_prf(y_test, pred, name, show=True)

In [ ]:
# per-FID maps for FIDs of your choice (must be test FIDs to show generalization)
ds = DisturbanceSegDataset("embeddings", os.path.expanduser("~/thesis_tiles_120px"),
                           "data_shp/label_polygons.shp")   # cheap

test_fids = np.load(f"{RESULT_DIRS['RandomForest']}/test_fids.npy")
print("test FIDs available:", sorted(set(test_fids.tolist()), key=int))
map_fids = sorted(set(test_fids.tolist()), key=int)[:4]   # edit to taste

for name, d in RESULT_DIRS.items():
    clf = joblib.load(f"{d}/model.joblib")
    ep.plot_fid_maps(ds, clf, map_fids, name, show=True)